# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [ ]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 9.1 MB/s eta 0:00:00


## 1) Define KB

In [ ]:
# Small knowledge base: 5 short snippets, each tagged with a source id
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


## 2) Define tools

In [ ]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = (
        "Looks up relevant snippets from a small custom knowledge base by keyword matching. "
        "Returns matching snippets prefixed with their source tag, e.g. [kb:agentic]."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query or question to look up in the knowledge base.",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "\n".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {
            "type": "string",
            "description": "The operation to perform: 'add' or 'multiply'. Defaults to 'add'.",
            "nullable": True,
        },
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()


## 3) Model (tiny local)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    model_id=MODEL_ID,
    max_new_tokens=200,
    device_map="auto",
)

print("Model ready:", MODEL_ID)


## 4) Agent

In [ ]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are a tiny agentic AI assistant. "
        "Use math_tool for arithmetic questions (addition or multiplication). "
        "Use kb_lookup_tool to search the knowledge base for conceptual questions. "
        "Keep answers concise (2-4 sentences). "
        "Always cite the knowledge base source tag (e.g., [kb:agentic]) when you use kb_lookup_tool. "
        "If no tool provides enough evidence, say so clearly and suggest a follow-up question."
    ),
)

print(agent)


## 5) Test queries

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)
